In [0]:
# ============================================================
# CELL 1: ADLS Connection + Config
# ============================================================

storage_account_name = "stsapadlsrawdev"
storage_account_key  = "YOUR_STORAGE_KEY_HERE"
silver_container     = "silver"
gold_container       = "gold"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

spark.conf.set("spark.databricks.io.cache.enabled",               "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",    "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",      "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print("ADLS connection configured!")
print("Delta Cache enabled!")
print("Auto Optimize enabled!")
print("Schema Evolution enabled!")

ADLS connection configured!
Delta Cache enabled!
Auto Optimize enabled!
Schema Evolution enabled!


In [0]:
# ============================================================
# CELL 2: Define Paths
# ============================================================

storage_account_name = "stsapadlsrawdev"
silver_container     = "silver"
gold_container       = "gold"

silver_path      = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
gold_path        = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

po_report_path   = f"{silver_path}/po_report/"
po_filtered_path = f"{gold_path}/po_filtered/"
watermark_path   = f"{gold_path}/po_filtered_watermark/"

print(f"Silver po_report  : {po_report_path}")
print(f"Gold po_filtered  : {po_filtered_path}")
print(f"Watermark         : {watermark_path}")

Silver po_report  : abfss://silver@stsapadlsrawdev.dfs.core.windows.net/po_report/
Gold po_filtered  : abfss://gold@stsapadlsrawdev.dfs.core.windows.net/po_filtered/
Watermark         : abfss://gold@stsapadlsrawdev.dfs.core.windows.net/po_filtered_watermark/


In [0]:
# ============================================================
# CELL 3: Setup Watermark Table (first run only)
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import col, current_timestamp, year, month, \
    max as spark_max, concat, lit, coalesce, trim

watermark_exists = DeltaTable.isDeltaTable(spark, watermark_path)

if not watermark_exists:
    schema = StructType([
        StructField("table_name",       StringType(),    False),
        StructField("last_recordstamp", TimestampType(), True)
    ])

    df_wm = spark.createDataFrame([
        ("po_filtered", None)
    ], schema)

    df_wm.write.format("delta").mode("overwrite").save(watermark_path)
    print("Watermark table created!")
else:
    print("Watermark table already exists!")

# Read current watermark
last_watermark = spark.read.format("delta") \
    .load(watermark_path) \
    .filter(col("table_name") == "po_filtered") \
    .collect()[0]["last_recordstamp"]

print(f"Current watermark: {last_watermark}")

Watermark table created!
Current watermark: None


In [0]:
# ============================================================
# CELL 4: Read po_report from Silver (Incremental)
# ============================================================

df_po_report = spark.read \
    .format("delta") \
    .load(po_report_path)

# Apply watermark filter
if last_watermark is not None:
    df_po_report = df_po_report.filter(
        col("source_recordstamp") > last_watermark
    )

print(f"po_report records to process: {df_po_report.count()}")
df_po_report.show(3)

po_report records to process: 3
+----------+-----------+---------+----------+---------+-------+--------+-----------+------------+--------------+----------------+----------------+--------------+-----------------+-------------+------------+---------------+--------------------+---------------+--------+---------+----------+-----+---------+--------------+-------------+-------------+----------------+-------+--------+--------------------+--------------------+
| po_number|vendor_name|vendor_id|   po_date|po_status|po_type|currency|po_category|company_code|purchasing_org|purchasing_group|release_strategy|release_status|release_indicator|payment_terms|po_line_item|material_number|material_description|unit_of_measure|quantity|net_price|price_unit|plant|net_value|material_group|deletion_flag|item_category|storage_location|po_year|po_month|  source_recordstamp|    silver_load_time|
+----------+-----------+---------+----------+---------+-------+--------+-----------+------------+--------------+------

In [0]:
# ============================================================
# CELL 5: Apply P2P Filters + Add PO_KEY
# Equivalent to TEMP_P2P_1_1 in your SQL
# ============================================================

df_po_filtered = df_po_report \
    .filter(
        # Filter 1: Remove STO orders
        ~col("po_type").like("%STO%")
    ) \
    .filter(
        # Filter 2: Remove deleted line items
        col("deletion_flag").isNull() | (col("deletion_flag") == "")
    ) \
    .filter(
        # Filter 3: Remove null or empty materials
        col("material_number").isNotNull() &
        (trim(col("material_number")) != "")
    ) \
    .filter(
        # Filter 4: Remove zero or null net price
        col("net_price").isNotNull() &
        (col("net_price") > 0)
    ) \
    .filter(
        # Filter 5: Remove zero or null quantity
        col("quantity").isNotNull() &
        (col("quantity") > 0)
    ) \
    .withColumn(
        "po_key",
        concat(
            trim(coalesce(col("material_number"), lit(""))),
            col("po_month").cast("string"),
            col("po_year").cast("string"),
            trim(coalesce(col("plant"), lit(""))),
            trim(coalesce(col("material_description"), lit("")))
        )
    ) \
    .withColumn("gold_load_time", current_timestamp())

print(f"Records after filters: {df_po_filtered.count()}")
df_po_filtered.select(
    "po_number", "po_line_item", "po_key",
    "material_number", "quantity", "net_price",
    "po_year", "po_month"
).show(5)

Records after filters: 3
+----------+------------+--------------------+---------------+--------+---------+-------+--------+
| po_number|po_line_item|              po_key|material_number|quantity|net_price|po_year|po_month|
+----------+------------+--------------------+---------------+--------+---------+-------+--------+
|4500000000|          10|C0000003120221000...|       C0000003|     1.0|      5.0|   2022|       1|
|4500000001|          10|C0000003120221000...|       C0000003|     5.0|     10.0|   2022|       1|
|4500000002|          10|TEST CPG112022100...|      TEST CPG1|     2.0|    100.0|   2022|       1|
+----------+------------+--------------------+---------------+--------+---------+-------+--------+



In [0]:
# ============================================================
# CELL 6: MERGE into po_filtered Delta
# ============================================================

po_filtered_exists = DeltaTable.isDeltaTable(spark, po_filtered_path)

if not po_filtered_exists:
    df_po_filtered.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("po_year", "po_month") \
        .option("mergeSchema", "true") \
        .save(po_filtered_path)

    spark.sql(f"""
        ALTER TABLE delta.`{po_filtered_path}`
        SET TBLPROPERTIES (
            'delta.dataSkippingNumIndexedCols' = '32',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    print("po_filtered Delta table created!")
else:
    delta_filtered = DeltaTable.forPath(spark, po_filtered_path)

    delta_filtered.alias("target").merge(
        df_po_filtered.alias("source"),
        """
        target.po_number    = source.po_number AND
        target.po_line_item = source.po_line_item
        """
    ).whenMatchedUpdate(
        condition = "source.source_recordstamp > target.source_recordstamp",
        set = {
            "po_key"               : "source.po_key",
            "vendor_id"            : "source.vendor_id",
            "vendor_name"          : "source.vendor_name",
            "po_date"              : "source.po_date",
            "po_status"            : "source.po_status",
            "po_type"              : "source.po_type",
            "currency"             : "source.currency",
            "po_category"          : "source.po_category",
            "company_code"         : "source.company_code",
            "purchasing_org"       : "source.purchasing_org",
            "purchasing_group"     : "source.purchasing_group",
            "release_strategy"     : "source.release_strategy",
            "release_status"       : "source.release_status",
            "release_indicator"    : "source.release_indicator",
            "payment_terms"        : "source.payment_terms",
            "material_number"      : "source.material_number",
            "material_description" : "source.material_description",
            "unit_of_measure"      : "source.unit_of_measure",
            "quantity"             : "source.quantity",
            "net_price"            : "source.net_price",
            "price_unit"           : "source.price_unit",
            "plant"                : "source.plant",
            "net_value"            : "source.net_value",
            "material_group"       : "source.material_group",
            "deletion_flag"        : "source.deletion_flag",
            "item_category"        : "source.item_category",
            "storage_location"     : "source.storage_location",
            "po_year"              : "source.po_year",
            "po_month"             : "source.po_month",
            "source_recordstamp"   : "source.source_recordstamp",
            "silver_load_time"     : "source.silver_load_time",
            "gold_load_time"       : "source.gold_load_time"
        }
    ).whenNotMatchedInsertAll() \
     .execute()

    print("po_filtered MERGE completed!")

total = spark.read.format("delta").load(po_filtered_path).count()
print(f"Total records in po_filtered: {total}")

po_filtered Delta table created!
Total records in po_filtered: 3


In [0]:
# ============================================================
# CELL 7: Update Watermark
# ============================================================

new_watermark = df_po_filtered \
    .agg(spark_max("source_recordstamp")) \
    .collect()[0][0]

print(f"New watermark: {new_watermark}")

if new_watermark is not None:
    delta_wm = DeltaTable.forPath(spark, watermark_path)

    delta_wm.alias("target").merge(
        spark.createDataFrame(
            [("po_filtered", new_watermark)],
            ["table_name", "last_recordstamp"]
        ).alias("source"),
        "target.table_name = source.table_name"
    ).whenMatchedUpdateAll() \
     .execute()

    print("Watermark updated!")
else:
    print("No new records — watermark unchanged!")

spark.read.format("delta").load(watermark_path).show()

New watermark: 2022-04-21 14:46:56.561898
Watermark updated!
+-----------+--------------------+
| table_name|    last_recordstamp|
+-----------+--------------------+
|po_filtered|2022-04-21 14:46:...|
+-----------+--------------------+



In [0]:
# ============================================================
# CELL 8: OPTIMIZE + Z-Order + VACUUM
# ============================================================

delta_filtered = DeltaTable.forPath(spark, po_filtered_path)

# Optimize + Z-Order by most queried columns
delta_filtered.optimize().executeZOrderBy(
    "vendor_id",
    "material_number",
    "po_key"
)
print("OPTIMIZE + Z-Order completed!")

# Vacuum old versions older than 7 days
delta_filtered.vacuum(168)
print("VACUUM completed!")

OPTIMIZE + Z-Order completed!
VACUUM completed!


In [0]:
# ============================================================
# CELL 9: Final Verification
# ============================================================

print("\n=== po_filtered Summary ===")

# Watermark
print("\n-- Watermark --")
spark.read.format("delta").load(watermark_path).show()

# Record count
total = spark.read.format("delta").load(po_filtered_path).count()
print(f"Total records: {total}")

# Sample data
print("\n-- Sample Data --")
spark.read.format("delta") \
    .load(po_filtered_path) \
    .select(
        "po_number", "po_line_item", "po_key",
        "material_number", "quantity", "net_price",
        "po_year", "po_month"
    ).show(5)

# Partitions
print("\n-- Partitions --")
files = dbutils.fs.ls(
    f"abfss://gold@{storage_account_name}.dfs.core.windows.net/po_filtered/"
)
for f in files:
    print(f.path)


=== po_filtered Summary ===

-- Watermark --
+-----------+--------------------+
| table_name|    last_recordstamp|
+-----------+--------------------+
|po_filtered|2022-04-21 14:46:...|
+-----------+--------------------+

Total records: 3

-- Sample Data --
+----------+------------+--------------------+---------------+--------+---------+-------+--------+
| po_number|po_line_item|              po_key|material_number|quantity|net_price|po_year|po_month|
+----------+------------+--------------------+---------------+--------+---------+-------+--------+
|4500000000|          10|C0000003120221000...|       C0000003|     1.0|      5.0|   2022|       1|
|4500000001|          10|C0000003120221000...|       C0000003|     5.0|     10.0|   2022|       1|
|4500000002|          10|TEST CPG112022100...|      TEST CPG1|     2.0|    100.0|   2022|       1|
+----------+------------+--------------------+---------------+--------+---------+-------+--------+


-- Partitions --
abfss://gold@stsapadlsrawdev.d